In [1]:
!pip install super_gradients==3.7.1 roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 19.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 18.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 14.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 17.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 15.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.5/684.5 kB 14.1 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 20.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 20

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="p5A1Dc89ug39Uh25pdHW")
project = rf.workspace("roboflow-58fyf").project("rock-paper-scissors-sxsw")
version = project.version(14)
dataset = version.download("darknet")

In [2]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130

/usr/bin/pip3:6: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import load_entry_point
Looking in indexes: https://download.pytorch.org/whl/cu130


In [3]:
import torch
print(torch.cuda.is_available())  # should be True
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(torch.cuda.current_device()))

True
0
NVIDIA GeForce RTX 5060 Ti


In [ ]:
from super_gradients.training.datasets import YoloDarknetFormatDetectionDataset

dataset_path = "./rock-paper-scissors-14"

train_dataset = YoloDarknetFormatDetectionDataset(
    data_dir=dataset_path,
    images_dir="train/",
    labels_dir="train/",
    classes=["Rock", "Paper", "Scissors"]
)

val_dataset = YoloDarknetFormatDetectionDataset(
    data_dir=dataset_path,
    images_dir="valid/",
    labels_dir="valid/",
    classes=["Rock", "Paper", "Scissors"]
)

[2026-03-04 12:44:20] INFO - crash_tips_setup.py - Crash tips is enabled. You can set your environment variable to CRASH_HANDLER=FALSE to disable it


The console stream is logged into /root/sg_logs/console.log
[WARNING]No module named 'pycocotools'


[2026-03-04 12:44:23] INFO - env_sanity_check.py - Library check is not supported when super_gradients installed through "git+https://github.com/..." command
[2026-03-04 12:44:23] INFO - detection_dataset.py - Dataset Initialization in progress. `cache_annotations=True` causes the process to take longer due to full dataset indexing.
Indexing dataset annotations: 100%|██████████| 6455/6455 [00:24<00:00, 266.74it/s]
[2026-03-04 12:44:47] INFO - detection_dataset.py - Dataset Initialization in progress. `cache_annotations=True` causes the process to take longer due to full dataset indexing.
Indexing dataset annotations:  92%|█████████▏| 529/576 [00:02<00:00, 255.81it/s]

Indexing dataset annotations: 100%|██████████| 576/576 [00:02<00:00, 264.98it/s]


In [5]:
# for x, y in train_dataset:
#     print(x.shape)
#     print(y.shape)

In [ ]:
import torch
print(torch.version.cuda)  # should be 13.x
print(torch.cuda.is_available())  # True
print(torch.cuda.get_device_name(0))  # Should print your RTX 5060 Ti

11.3
True
NVIDIA GeForce RTX 5060 Ti


In [ ]:
from super_gradients.training import models
from super_gradients.training import Trainer
import torch
from torch.utils.data import DataLoader
from super_gradients.common.object_names import Models

trainer = Trainer(experiment_name="rps_yolonas", ckpt_root_dir="./checkpoints")

model = models.get(
    Models.YOLOX_N,
    num_classes=3,
    checkpoint_path="/app/models/yolox_nano.pth",
    strict_load=False
)

def collate_fn(batch):
    images, targets = zip(*batch)
    
    # Stack images normally
    images = torch.stack([torch.tensor(img).permute(2, 0, 1).float() for img in images])
    
    # Keep targets as list of tensors (variable length)
    targets = [torch.tensor(t) for t in targets]
    
    return images, targets

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16,collate_fn=collate_fn)

trainer.train(
    model=model,
    train_loader=train_loader,
    valid_loader=val_loader,
    training_params={
        "max_epochs": 30,
        "initial_lr": 5e-4,
        "lr_mode": "cosine",
        "loss": "CrossEntropyLoss",  # <-- string name also works
    }
)

/usr/local/lib/python3.8/dist-packages/super_gradients/common/deprecate.py:279: DeprecationWarning: Parameter `arch_params.channels_in` is deprecated since version `3.3.0` and will be removed in version `4.0.0`.
Please update your code to use the `arch_params.in_channels` instead of `arch_params.channels_in`.
  warnings.warn(message, DeprecationWarning)
[2026-03-04 12:44:57] INFO - checkpoint_utils.py - Successfully loaded model weights from /app/models/yolox_nano.pth checkpoint.
[2026-03-04 12:44:57] WARNING - sg_trainer.py - Train dataset size % batch_size != 0 and drop_last=False, this might result in smaller last batch.
[2026-03-04 12:44:57] INFO - sg_trainer.py - Starting a new run with `run_id=RUN_20260304_124457_865677`
[2026-03-04 12:44:57] INFO - sg_trainer.py - Checkpoints directory: ./checkpoints/rps_yolonas/RUN_20260304_124457_865677
/usr/local/lib/python3.8/dist-packages/super_gradients/training/utils/optimizer_utils.py:108: DeprecationWarning: initialize_param_groups and 

The console stream is now moved to ./checkpoints/rps_yolonas/RUN_20260304_124457_865677/console_Mar04_12_44_57.txt


[2026-03-04 12:44:58] INFO - sg_trainer_utils.py - TRAINING PARAMETERS:
    - Mode:                         Single GPU
    - Number of GPUs:               1          (1 available on the machine)
    - Full dataset size:            3939       (len(train_set))
    - Batch size per GPU:           16         (batch_size)
    - Batch Accumulate:             1          (batch_accumulate)
    - Total batch size:             16         (num_gpus * batch_size)
    - Effective Batch size:         16         (num_gpus * batch_size * batch_accumulate)
    - Iterations per epoch:         247        (len(train_loader))
    - Gradient updates per epoch:   247        (len(train_loader) / batch_accumulate)
    - Model: YoloX_N  (897.14K parameters, 897.14K optimized)
    - Learning Rates and Weight Decays:
      - default: (897.14K parameters). LR: 0.0005 (897.14K parameters) WD: 0.0001, (897.14K parameters)

[2026-03-04 12:44:58] INFO - sg_trainer.py - Started training for 30 epochs (0/29)

Train epoc

RuntimeError: cuDNN error: CUDNN_STATUS_NOT_INITIALIZED